# Olist Data Understanding

This notebook answers the prerequisite questions before any modeling:

- What is the grain of each raw table?
- How do the tables join together, and what cardinality should we expect?
- Where is data missing?
- Are there duplicate keys beyond the declared ones?
- How many orders are we losing at each step to reach the delivered-only modeling subset?

It is deliberately scoped to **data understanding only** - no features, no models, no plots beyond what a funnel needs. Modeling-oriented EDA lives in `02_eda.ipynb`.

In [36]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.io import load_olist_tables

pd.set_option("display.max_columns", 50)

tables = load_olist_tables(ROOT / "data" / "raw")
list(tables.keys())

['customers', 'orders', 'order_items', 'order_payments', 'products', 'sellers']

## 1. Table grain

Each row in the Olist tables represents a different unit. Before joining anything, confirm the declared primary key and verify it is actually unique.

In [37]:
declared_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}

grain_rows = []
for name, df in tables.items():
    key = declared_keys[name]
    dupes = int(df.duplicated(subset=key).sum())
    grain_rows.append({
        "table": name,
        "rows": len(df),
        "declared_key": " + ".join(key),
        "key_is_unique": dupes == 0,
        "duplicate_rows_on_key": dupes,
    })

pd.DataFrame(grain_rows)

,table,rows,declared_key,key_is_unique,duplicate_rows_on_key
0,customers,99441,customer_id,True,0
1,orders,99441,order_id,True,0
2,order_items,112650,order_id + order_item_id,True,0
3,order_payments,103886,order_id + payment_sequential,True,0
4,products,32951,product_id,True,0
5,sellers,3095,seller_id,True,0


## 2. Join cardinality

The modeling frame starts from `orders` and left-joins aggregates from the other tables. Each relationship should match a specific expected shape:

- `orders` <-> `customers`: **1:1** (one customer per order)
- `orders` <-> `order_items`: **1:N** (an order can contain multiple items)
- `orders` <-> `order_payments`: **1:N** (split payments are allowed)
- `order_items` <-> `products` / `sellers`: **N:1** from the items side

If any right-hand key fans out unexpectedly, a naive merge on `orders` would silently multiply rows.

In the actual data, `orders -> order_items` leaves 775 unmatched orders and `orders -> order_payments` leaves 1 unmatched order. Because the modeling frame uses left joins, those orders stay in the order-level dataset; the missing aggregates are later imputed rather than dropped.

In [38]:
orders = tables["orders"]
customers = tables["customers"]
order_items = tables["order_items"]
order_payments = tables["order_payments"]
products = tables["products"]
sellers = tables["sellers"]


def summarize_join(left, right, on, left_name, right_name):
    left_keys = left[[on]]
    unique_right_keys = right[[on]].drop_duplicates()
    merged = left_keys.merge(unique_right_keys, on=on, how="left", indicator=True)
    unmatched = int((merged["_merge"] == "left_only").sum())
    right_counts = right.groupby(on).size()
    return {
        "join": f"{left_name} -> {right_name}",
        "on": on,
        "left_rows": len(left),
        "unmatched_left_rows": unmatched,
        "max_right_per_key": int(right_counts.max()) if len(right_counts) else 0,
        "mean_right_per_key": round(float(right_counts.mean()), 2) if len(right_counts) else 0.0,
    }


pd.DataFrame([
    summarize_join(orders, customers, "customer_id", "orders", "customers"),
    summarize_join(orders, order_items, "order_id", "orders", "order_items"),
    summarize_join(orders, order_payments, "order_id", "orders", "order_payments"),
    summarize_join(order_items, products, "product_id", "order_items", "products"),
    summarize_join(order_items, sellers, "seller_id", "order_items", "sellers"),
])

,join,on,left_rows,unmatched_left_rows,max_right_per_key,mean_right_per_key
0,orders -> customers,customer_id,99441,0,1,1.00
1,orders -> order_items,order_id,99441,775,21,1.14
2,orders -> order_payments,order_id,99441,1,29,1.04
3,order_items -> products,product_id,112650,0,1,1.00
4,order_items -> sellers,seller_id,112650,0,1,1.00


## 3. Missingness

Focus on columns that affect the target (`orders` timestamps) or enter the feature set. Only columns with at least one null are shown.

In [39]:
def missingness(df, table_name):
    total = len(df)
    missing = df.isna().sum()
    frame = pd.DataFrame({
        "table": table_name,
        "column": missing.index,
        "missing": missing.values,
        "missing_pct": (missing.values / total * 100).round(2),
    })
    return frame[frame["missing"] > 0]


missing_report = pd.concat(
    [missingness(df, name) for name, df in tables.items()],
    ignore_index=True,
).sort_values(["table", "missing_pct"], ascending=[True, False])
missing_report

,table,column,missing,missing_pct
2,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
0,orders,order_approved_at,160,0.16
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,products,product_weight_g,2,0.01
8,products,product_length_cm,2,0.01
9,products,product_height_cm,2,0.01


## 4. Duplicate checks beyond primary keys

The declared keys above confirm row uniqueness, but it is still worth checking whether the same `order_id` ever shows up with conflicting customer or status information. If so, downstream joins could silently fan out or the target could become ambiguous.

In [40]:
order_customer_consistency = orders.groupby("order_id")["customer_id"].nunique()
order_status_consistency = orders.groupby("order_id")["order_status"].nunique()

pd.DataFrame([
    {"check": "orders with >1 customer_id", "count": int((order_customer_consistency > 1).sum())},
    {"check": "orders with >1 order_status", "count": int((order_status_consistency > 1).sum())},
    {"check": "exact duplicate rows in orders", "count": int(orders.duplicated().sum())},
    {"check": "exact duplicate rows in order_items", "count": int(order_items.duplicated().sum())},
    {"check": "exact duplicate rows in order_payments", "count": int(order_payments.duplicated().sum())},
])

,check,count
0,orders with >1 customer_id,0
1,orders with >1 order_status,0
2,exact duplicate rows in orders,0
3,exact duplicate rows in order_items,0
4,exact duplicate rows in order_payments,0


## 5. Order-status funnel to the modeling subset

The modeling frame uses only delivered orders with non-null delivery and estimated-delivery timestamps. This funnel shows exactly how many orders are dropped at each gate, so the delivered-only subset is an auditable decision rather than an implicit one.

In [41]:
total_orders = len(orders)
by_status = (
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="n_orders")
)
by_status["share_pct"] = (by_status["n_orders"] / total_orders * 100).round(2)
by_status

,order_status,n_orders,share_pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [42]:
def funnel_row(label, df):
    return {
        "stage": label,
        "n_orders": len(df),
        "share_of_raw_pct": round(len(df) / total_orders * 100, 2),
    }


stage1 = orders
stage2 = stage1[stage1["order_status"] == "delivered"]
stage3 = stage2.dropna(subset=["order_delivered_customer_date"])
stage4 = stage3.dropna(subset=["order_estimated_delivery_date"])
stage5 = stage4.dropna(subset=["order_purchase_timestamp"])

pd.DataFrame([
    funnel_row("1. all raw orders", stage1),
    funnel_row("2. status == delivered", stage2),
    funnel_row("3. + has delivered_customer_date", stage3),
    funnel_row("4. + has estimated_delivery_date", stage4),
    funnel_row("5. + has purchase_timestamp", stage5),
])

,stage,n_orders,share_of_raw_pct
0,1. all raw orders,99441,100.00
1,2. status == delivered,96478,97.02
2,3. + has delivered_customer_date,96470,97.01
3,4. + has estimated_delivery_date,96470,97.01
4,5. + has purchase_timestamp,96470,97.01


## Decisions taken into modeling

- **Grain.** The modeling frame is one row per `order_id`. Item, payment, product, and seller information is aggregated to that grain before any join touches `orders`.
- **Cardinality surprise.** An order can contain multiple items, multiple payments, and items from multiple sellers. All three need explicit aggregation (count, sum, mode) before joining - otherwise a naive merge fans the orders frame out.
- **Join gaps.** A small set of orders has no matching item or payment rows. Left joins preserve them in the modeling frame, and downstream aggregate columns are imputed rather than causing silent row drops.
- **Missingness.** The delivery timestamps are missing on a predictable subset (non-delivered orders) rather than at random. Product metadata missingness is tracked as a feature (`missing_product_metadata_share`) rather than imputed blindly.
- **Subset filter.** Only rows reaching the final funnel stage enter the model. The dropped rows are cancellations, unavailable stock, and orders still in transit - their exclusion is a **scope decision**, not a cleaning decision, and it means the model predicts late-vs-on-time only for orders that eventually reach a delivered state.

For distribution plots, segment-level late rates, and feature-level exploration, see `02_eda.ipynb`.